# Raw ridership: ingestion and quality

## Summary
2,800 daily rows, 14 services, no duplicate dates or date gaps. There are 12,193 unreported service-day cells, 290 zeros, and no negative values. Coverage differences matter more than row-level cleaning in this snapshot.

## Context and methods
The supplied CSV is the controlling input. [Official catalogue](https://data.gov.my/data-catalogue/ridership_headline), checked 2026-10-05; its published data end date matches this snapshot. Counts represent headline rides, not unique passengers. No refresh or raw edits are made.

### Assumptions
Blank counts are unknown, not zero. A first observation is a reporting boundary, not proof of a service opening. One extract cannot establish schema history or backfills.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.config import RAW_PATH, OUTPUT_DIR, SERVICES, CORE_SERVICES
from src.data_cleaning import clean_data, quality_profile, reshape_data, validate_outputs
pd.set_option('display.max_rows', 16)
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 120)


## Data
Check the source dimensions, schema, types and boundaries before making transformations.

In [2]:
raw = pd.read_csv(RAW_PATH)
dates = pd.to_datetime(raw['date'], format='%Y-%m-%d')
display(pd.Series({'rows': len(raw), 'columns': len(raw.columns), 'first_date': dates.min(), 'last_date': dates.max(), 'duplicate_rows': raw.duplicated().sum(), 'duplicate_dates': dates.duplicated().sum(), 'date_gaps': len(pd.date_range(dates.min(), dates.max()).difference(dates))}).to_frame('value'))
display(raw.dtypes.to_frame('raw_dtype'))
display(raw.head(3))

,value
rows,2800
columns,15
first_date,2019-01-01 00:00:00
last_date,2026-08-31 00:00:00
duplicate_rows,0
duplicate_dates,0
date_gaps,0


,raw_dtype
date,object
bus_rkl,float64
bus_rkn,float64
bus_rpn,float64
rail_lrt_ampang,int64
rail_mrt_kajang,int64
rail_lrt_kj,int64
rail_monorail,int64
rail_mrt_pjy,float64
rail_lrt_shah_alam,float64


,date,bus_rkl,bus_rkn,bus_rpn,rail_lrt_ampang,...,rail_ets,rail_intercity,rail_komuter_utara,rail_tebrau,rail_komuter
0,2019-01-01,NaN,NaN,NaN,113357,...,NaN,NaN,NaN,NaN,NaN
1,2019-01-02,NaN,NaN,NaN,182715,...,NaN,NaN,NaN,NaN,NaN
2,2019-01-03,NaN,NaN,NaN,187904,...,NaN,NaN,NaN,NaN,NaN


## Results
Compare missing counts, zeros and internal gaps. Zero counts remain valid source observations, but long runs need follow-up before interpreting them as demand.

In [3]:
profile = quality_profile(raw)
display(profile[['service_code', 'observed_days', 'missing_pct', 'zero_days', 'internal_missing_days']].round(1))

,service_code,observed_days,missing_pct,zero_days,internal_missing_days
0,bus_rkl,1704,39.1,0,0
1,bus_rkn,1704,39.1,260,0
2,bus_rpn,1704,39.1,0,0
3,rail_lrt_ampang,2800,0.0,0,0
4,rail_mrt_kajang,2800,0.0,0,0
5,rail_lrt_kj,2800,0.0,0,0
6,rail_monorail,2800,0.0,0,0
7,rail_mrt_pjy,1538,45.1,0,0
8,rail_lrt_shah_alam,94,96.6,30,59
9,rail_ets,2147,23.3,0,0


### Reporting boundaries and value ranges

In [4]:
display(profile[['service_code', 'first_observed', 'last_observed', 'minimum', 'maximum']])
assert profile.negative_days.sum() == 0
assert set(raw.columns) == {'date', *SERVICES}

,service_code,first_observed,last_observed,minimum,maximum
0,bus_rkl,2022-01-01,2026-08-31,71611.0,269734.0
1,bus_rkn,2022-01-01,2026-08-31,0.0,4865.0
2,bus_rpn,2022-01-01,2026-08-31,20758.0,60669.0
3,rail_lrt_ampang,2019-01-01,2026-08-31,6587.0,270317.0
4,rail_mrt_kajang,2019-01-01,2026-08-31,4973.0,401756.0
5,rail_lrt_kj,2019-01-01,2026-08-31,7195.0,352504.0
6,rail_monorail,2019-01-01,2026-08-31,1392.0,99361.0
7,rail_mrt_pjy,2022-06-16,2026-08-31,12108.0,236225.0
8,rail_lrt_shah_alam,2026-04-01,2026-08-31,0.0,110341.0
9,rail_ets,2020-10-15,2026-08-31,50.0,25781.0


### Suspicious zero runs
The two services below account for all source zeros. Reporting zeros and reporting blanks are kept separate.

In [5]:
zero_ranges = []
for code in ['bus_rkn', 'rail_lrt_shah_alam']:
    zero = raw[code].eq(0)
    groups = zero.ne(zero.shift()).cumsum()
    for _, indexes in raw[zero].groupby(groups[zero]).groups.items():
        zero_ranges.append({'service': SERVICES[code][0], 'start': dates.loc[indexes].min(), 'end': dates.loc[indexes].max(), 'days': len(indexes)})
display(pd.DataFrame(zero_ranges).sort_values('days', ascending=False).head(10))

,service,start,end,days
0,Rapid Bus Kuantan,2025-12-15,2026-08-31,260
1,LRT Shah Alam,2026-04-01,2026-04-30,30


### Daily spikes and drops
Compare each service with its preceding eight matching weekdays. The full criterion is documented in data/README.md. This is a screening list; counts are not deleted.

In [6]:
from src.features import anomaly_flags
flags = anomaly_flags(reshape_data(clean_data(raw)))
candidates = flags[flags.is_anomaly].sort_values('anomaly_score', key=abs, ascending=False)
display(candidates[['date', 'service', 'ridership', 'weekday_baseline', 'anomaly_score']].head(10).round(2))
print(f'{len(candidates):,} review candidates; no automatic removals')

,date,service,ridership,weekday_baseline,anomaly_score
25193,2026-08-25,LRT Kelana Jaya,143736,305489.88,-67.93
35138,2023-03-19,MRT Putrajaya,126977,14987.50,63.82
35137,2023-03-18,MRT Putrajaya,127158,15994.88,60.48
16588,2026-02-01,KTM Komuter,87165,19859.38,48.72
24835,2025-09-01,LRT Kelana Jaya,136334,292303.75,-48.63
9419,2021-10-16,KTMB ETS,3059,228.38,46.66
16587,2026-01-31,KTM Komuter,72501,19650.75,44.48
33593,2026-08-25,MRT Kajang,184354,308778.50,-43.97
2439,2025-09-05,Rapid Bus KL,159350,237976.00,-43.83
36303,2026-05-27,MRT Putrajaya,106140,197797.62,-43.64


684 review candidates; no automatic removals


## Takeaways
The source supports service-level demand analysis, with explicit coverage restrictions. Keep nulls and zeros intact. Use the four continuously reported rail services for long-term comparisons; require complete service-period coverage for growth. Verify zero runs and extreme dates with operator records before assigning causes. No historical schema-change claim can be made from this single snapshot.